# RAG + LLM Evaluation Runner (Clean)

This notebook runs RAG+LLM evaluation using only `query_strategy_regex_intelligent_v2`.

Flow:
1. Load evaluation records and config.
2. Build query text with regex-intelligent v2.
3. Precompute and save retrieval chunk caches for:
   - `baseline` (raw retrieval order)
   - `balanced` (reranked selection)
4. Reuse selected cache mode during LLM inference (no retrieval at API-call time).

In [ ]:
import os, json, time
from dotenv import load_dotenv
from pathlib import Path
from groq import Groq
from concurrent.futures import ThreadPoolExecutor, as_completed

ROOT = Path("..").resolve()
EVAL_PATH = ROOT / 'data' / 'processed' / 'evaluation.json'
OUT_DIR = ROOT / 'outputs' 
RAW_OUT = OUT_DIR / 'rag_results' / 'rag_llm_raw_responses_prompt_v3.txt'
ZERO_RESPONSE_PR = OUT_DIR / 'rag_results' / 'zero_response_PRs_rag_v3.txt'
PARSED_OUT = OUT_DIR / 'rag_results' /  'llm_reviews.json'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL = 'openai/gpt-oss-20b'
BATCH_SIZE = 1
MAX_CHARS_PER_FILE = 8000

# evaluation.json range controls (inclusive, 0-based)
STARTING = 0
ENDING = 96

# Token controls
MODEL_CONTEXT_LIMIT = 8192
MAX_OUTPUT_TOKENS = 800
INPUT_TOKEN_BUDGET = int(MODEL_CONTEXT_LIMIT * 0.75) - MAX_OUTPUT_TOKENS
CHARS_PER_TOKEN = 2.5

# Retrieval controls
RETRIEVAL_TOP_K = 7
RETRIEVAL_MODE = 'balanced'  # one of: 'baseline', 'balanced'
BASELINE_CACHE_PATH = OUT_DIR / f'baseline_chunks_top_k_{RETRIEVAL_TOP_K}.json'
BALANCED_CACHE_PATH = OUT_DIR / f'balanced_chunks_top_k_{RETRIEVAL_TOP_K}.json'
PROMPT_PATH = ROOT / 'src' / 'rag_model' / 'prompts' / 'v3.txt'
PROMPT_TEMPLATE = PROMPT_PATH.read_text(encoding='utf-8')

load_dotenv()
GROQ_API_KEY = os.environ['GROQ_API_KEY_V2']
GROQ_API_URL = os.environ.get('GROQ_API_URL', 'https://api.groq.com')
client = Groq(api_key=GROQ_API_KEY, base_url=GROQ_API_URL)

In [2]:
import sys
sys.path.insert(0, str(ROOT / 'src'))

from rag_model.query_strategy import (
    query_strategy_regex_intelligent_v2,
    retrieve_guidelines,
    payload_to_text,
)

In [3]:
def load_evaluation(path: Path):
    data = json.loads(path.read_text(encoding='utf-8'))
    return [r for r in data if isinstance(r, dict) and r.get('source_file')]


def resolve_source_path(source_file: str):
    p = Path(source_file)
    if p.is_absolute() and p.exists():
        return p
    p1 = ROOT / source_file
    if p1.exists():
        return p1
    p2 = ROOT / 'data' / 'processed' / source_file
    if p2.exists():
        return p2
    p3 = ROOT / 'data' / 'processed' / 'evaluation_files' / p.name
    return p3


def read_source_text(source_file: str):
    p = resolve_source_path(source_file)
    text = p.read_text(encoding='utf-8', errors='ignore')
    if len(text) <= MAX_CHARS_PER_FILE:
        return text
    half = MAX_CHARS_PER_FILE // 2
    return text[:half] + '\n\n...TRUNCATED...\n\n' + text[-half:]


def est_tokens(text: str):
    return max(1, int(len(text) / CHARS_PER_TOKEN))


def _normalize_category(raw):
    if not isinstance(raw, str):
        return None
    value = raw.strip().lower().replace('-', '_').replace(' ', '_')
    return value if value else None


def _extract_category(payload):
    if not isinstance(payload, dict):
        return None
    for key in ['violation_category', 'category', 'label', 'violation']:
        cat = _normalize_category(payload.get(key))
        if cat:
            return cat
    return None


def _build_candidate_rows(points, query_text):
    query_text_lc = str(query_text).lower()
    query_tokens = set(query_text_lc.replace('_', ' ').split())
    rows = []

    for rank_idx, point in enumerate(points):
        payload = getattr(point, 'payload', None)
        text_val = payload_to_text(payload)
        category = _extract_category(payload)
        sem_score = float(getattr(point, 'score', 0.0) or 0.0)
        chunk_tokens = set(str(text_val).lower().replace('_', ' ').split())
        lexical_overlap = len(query_tokens & chunk_tokens) / max(1, len(query_tokens))
        is_category_match = int(bool(category and category in query_text_lc))

        rows.append({
            'rank_idx': rank_idx,
            'category': category,
            'sem_score': sem_score,
            'lexical_overlap': lexical_overlap,
            'is_category_match': is_category_match,
            'text': text_val,
            'payload': payload,
        })
    return rows


def _build_query_entry(entry: dict):
    return {
        'file_text': entry.get('source_code', ''),
        'source_file': entry.get('source_file'),
        'repo': entry.get('repo'),
    }


def build_baseline_context(entry: dict):
    query_text = query_strategy_regex_intelligent_v2(_build_query_entry(entry))
    points = retrieve_guidelines(
        query_text=query_text,
        repo_name=entry.get('repo'),
        top_k=RETRIEVAL_TOP_K,
    )
    rows = _build_candidate_rows(points, query_text)
    chunks = [r['text'] for r in rows if r['text']][:RETRIEVAL_TOP_K]

    candidates = []
    for r in rows[:RETRIEVAL_TOP_K]:
        candidates.append({
            'rank_idx': r['rank_idx'],
            'category': r['category'],
            'score': r['sem_score'],
            'text': r['text'],
            'payload': r['payload'],
        })
    return query_text, chunks, candidates


def build_balanced_context(entry: dict):
    query_text = query_strategy_regex_intelligent_v2(_build_query_entry(entry))

    top_n_candidates = max(25, RETRIEVAL_TOP_K * 4)
    points = retrieve_guidelines(
        query_text=query_text,
        repo_name=entry.get('repo'),
        top_k=top_n_candidates,
    )
    rows = _build_candidate_rows(points, query_text)

    lexical_weight = 0.35
    category_bonus = 0.15
    rank_penalty = 0.01
    max_per_category = 2

    for row in rows:
        row['rerank_score'] = (
            row['sem_score']
            + (lexical_weight * row['lexical_overlap'])
            + (category_bonus * row['is_category_match'])
            - (rank_penalty * row['rank_idx'])
        )

    reranked_rows = []
    cat_counts = {}
    for row in sorted(rows, key=lambda x: x['rerank_score'], reverse=True):
        if not row['text']:
            continue
        category = row['category']
        if category:
            current_count = cat_counts.get(category, 0)
            if current_count >= max_per_category:
                continue
            cat_counts[category] = current_count + 1
        reranked_rows.append(row)
        if len(reranked_rows) >= RETRIEVAL_TOP_K:
            break

    chunks = [r['text'] for r in reranked_rows]
    candidates = []
    for r in reranked_rows:
        candidates.append({
            'rank_idx': r['rank_idx'],
            'category': r['category'],
            'score': r['sem_score'],
            'rerank_score': r['rerank_score'],
            'text': r['text'],
            'payload': r['payload'],
        })
    return query_text, chunks, candidates


def save_chunk_cache(prepared_entries, cache_path: Path, mode: str):
    cache_data = []
    for entry in prepared_entries:
        cache_data.append({
            'id': entry['id'],
            'mode': mode,
            'top_k': RETRIEVAL_TOP_K,
            'query_text': entry.get('query_text', ''),
            'retrieved_chunks': entry.get('retrieved_chunks', []),
            'retrieved_candidates': entry.get('retrieved_candidates', []),
        })
    cache_path.write_text(json.dumps(cache_data, indent=2), encoding='utf-8')


def load_chunk_cache(cache_path: Path):
    if not cache_path.exists():
        return {}
    cache_data = json.loads(cache_path.read_text(encoding='utf-8'))
    result = {}
    for item in cache_data:
        if not isinstance(item, dict):
            continue
        result[item.get('id')] = item
    return result


records = load_evaluation(EVAL_PATH)
records = records[STARTING:ENDING + 1]
print('selected_records:', len(records), 'range:', STARTING, 'to', ENDING, '(inclusive)')
print('retrieval_top_k:', RETRIEVAL_TOP_K)
print('baseline_cache:', BASELINE_CACHE_PATH)
print('balanced_cache:', BALANCED_CACHE_PATH)

selected_records: 97 range: 0 to 96 (inclusive)
retrieval_top_k: 7
baseline_cache: C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project\outputs\baseline_chunks_top_k_7.json
balanced_cache: C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project\outputs\balanced_chunks_top_k_7.json


In [4]:
def build_prompt(batch):
    header = PROMPT_TEMPLATE.strip()

    blocks = [header]
    for item in batch:
        chunks = item.get('retrieved_chunks', [])
        chunk_block = '\n\n'.join([f'[Chunk {idx+1}]\n{c}' for idx, c in enumerate(chunks)])
        if not chunk_block:
            chunk_block = '[No retrieved payload chunks available]'

        blocks.append(
f"""RETRIEVED_PAYLOAD_CHUNKS:
{chunk_block}
END

PR:
ID: {item['id']}
CODE:
{item['source_code']}
"""
        )

    return '\n'.join(blocks)

def call_groq(prompt):
    return client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[{'role': 'user', 'content': prompt}]
    )

In [5]:
batch_size = BATCH_SIZE


def prepare_entries_for_mode(records_subset, mode: str, cache_path: Path):
    if mode not in {'baseline', 'balanced'}:
        raise ValueError(f'Unsupported mode: {mode}')

    cache_by_id = load_chunk_cache(cache_path)
    prepared_entries = []
    records_to_fetch = []

    for r in records_subset:
        cached = cache_by_id.get(r['id'])
        if (
            isinstance(cached, dict)
            and cached.get('mode') == mode
            and cached.get('top_k') == RETRIEVAL_TOP_K
            and isinstance(cached.get('retrieved_chunks'), list)
            and len(cached.get('retrieved_chunks')) > 0
        ):
            entry = {
                'id': r['id'],
                'repo': r.get('repo'),
                'source_file': r['source_file'],
                'source_code': read_source_text(r['source_file']),
                'query_text': cached.get('query_text', ''),
                'retrieved_chunks': cached.get('retrieved_chunks', []),
                'retrieved_candidates': cached.get('retrieved_candidates', []),
            }
            prepared_entries.append(entry)
        else:
            records_to_fetch.append(r)

    print(f'[{mode}] using {len(prepared_entries)} cached entries, fetching {len(records_to_fetch)} new entries')

    if records_to_fetch:
        builder = build_baseline_context if mode == 'baseline' else build_balanced_context

        with ThreadPoolExecutor(max_workers=4) as executor:
            future_to_record = {}
            for r in records_to_fetch:
                entry_partial = {
                    'id': r['id'],
                    'repo': r.get('repo'),
                    'source_file': r['source_file'],
                    'source_code': read_source_text(r['source_file']),
                }
                future = executor.submit(builder, entry_partial)
                future_to_record[future] = entry_partial

            for i, future in enumerate(as_completed(future_to_record), start=1):
                entry_partial = future_to_record[future]
                try:
                    qtext, chunks, candidates = future.result()
                    entry_partial['query_text'] = qtext
                    entry_partial['retrieved_chunks'] = chunks
                    entry_partial['retrieved_candidates'] = candidates
                except Exception as e:
                    entry_partial['query_text'] = f'<query/retrieval error: {e}>'
                    entry_partial['retrieved_chunks'] = []
                    entry_partial['retrieved_candidates'] = []
                    print(f'  [{mode}] error on record {entry_partial["id"]}: {e}')

                prepared_entries.append(entry_partial)
                if i % 10 == 0 or i == len(records_to_fetch):
                    print(f'  [{mode}] completed {i}/{len(records_to_fetch)} retrieval builds')

        # Persist full mode cache after fresh fetches.
        save_chunk_cache(prepared_entries, cache_path, mode)
        print(f'[{mode}] cache updated: {cache_path}')

    # Keep deterministic ordering by id for stable batching.
    prepared_entries.sort(key=lambda x: x['id'])
    return prepared_entries


# Precompute and persist both cache files (baseline + balanced) once per run.
prepared_baseline = prepare_entries_for_mode(records, 'baseline', BASELINE_CACHE_PATH)
prepared_balanced = prepare_entries_for_mode(records, 'balanced', BALANCED_CACHE_PATH)

# Select which precomputed cache to use for LLM inference.
if RETRIEVAL_MODE == 'baseline':
    prepared = prepared_baseline
elif RETRIEVAL_MODE == 'balanced':
    prepared = prepared_balanced
else:
    raise ValueError(f'Unknown RETRIEVAL_MODE={RETRIEVAL_MODE}. Use baseline or balanced.')

print(f'Running inference with mode={RETRIEVAL_MODE}, prepared entries={len(prepared)}')

id_to_repo = {x['id']: x['repo'] for x in prepared}
zero_response_prs = set()

i = 0
batch_no = 0
while i < len(prepared):
    time.sleep(2)
    batch = []
    while i < len(prepared) and len(batch) < batch_size:
        batch.append(prepared[i])
        i += 1

    batch_no += 1
    prompt = build_prompt(batch)
    resp = call_groq(prompt)
    content = resp.choices[0].message.content

    print(f'\n=== BATCH {batch_no} RESPONSE DIAGNOSTICS ===')
    pr_ids = [p['id'] for p in batch]
    print(f'PR ids: {pr_ids}')
    print(f'content_type: {type(content).__name__}')

    is_empty = False
    if isinstance(content, str):
        content_stripped = content.strip()
        print(f'content_len: {len(content)}')
        print(f'is_empty_after_strip: {len(content_stripped) == 0}')
        print('response_preview:')
        print(content[:1200])
        if len(content_stripped) == 0:
            is_empty = True
    else:
        print('response_preview_non_str:')
        print(content)
        is_empty = True

    if is_empty:
        print(f'Empty response detected for batch {batch_no}')
        zero_response_prs.update(pr_ids)

    content_to_write = content if isinstance(content, str) else str(content)

    # Keep exact header/body format used in naive_llm
    with open(RAW_OUT, 'a', encoding='utf-8') as rf:
        rf.write(f'Batch {batch_no} - PR ids: {pr_ids}\n')
        rf.write(f'repos: {[id_to_repo[p_id] for p_id in pr_ids]}\n')
        rf.write(content_to_write)
        rf.write('\n\n\n')

    print(f'Batch {batch_no}: {len(batch)} PRs, est_input_tokens={est_tokens(prompt)}')

if zero_response_prs:
    with open(ZERO_RESPONSE_PR, 'w', encoding='utf-8') as f:
        for pr_id in sorted(zero_response_prs):
            f.write(f'{pr_id}\n')
    print(f'Wrote {len(zero_response_prs)} zero-response PR ids to {ZERO_RESPONSE_PR}')
else:
    print('No zero-response PRs detected.')

[baseline] using 97 cached entries, fetching 0 new entries
[balanced] using 97 cached entries, fetching 0 new entries
Running inference with mode=balanced, prepared entries=97

=== BATCH 1 RESPONSE DIAGNOSTICS ===
PR ids: ['synthetic-django_PR_21']
content_type: str
content_len: 424
is_empty_after_strip: False
response_preview:
[
  {
    "PR_ID": "synthetic-django_PR_21",
    "llm_reviews": [
      {
        "line_number": 9,
        "violation_category": "unused_import",
        "review_comment": "Remove unused import 'os'."
      },
      {
        "line_number": 10,
        "violation_category": "unused_import",
        "review_comment": "Remove unused import 'sys'."
      },
      {
        "line_number": 11,
        "violation_category": "
Batch 1: 1 PRs, est_input_tokens=2043

=== BATCH 2 RESPONSE DIAGNOSTICS ===
PR ids: ['synthetic-django_PR_22']
content_type: str
content_len: 466
is_empty_after_strip: False
response_preview:
[
  {
    "PR_ID": "synthetic-django_PR_22",
    "llm